**P1-Supply Chain Monitoring
and Optimization Platform**

**Week 3 – Intro to PySpark: Processing Big Data**

In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("P1_Week3").getOrCreate()

In [3]:
from pyspark.sql import functions as F

In [5]:
from google.colab import files
uploaded = files.upload()

Saving suppliers.csv to suppliers.csv


**1.Load order data from CSV using PySpark.**

In [6]:
orders = spark.read.csv("cleaned_orders.csv", header=True, inferSchema=True)
suppliers = spark.read.csv("suppliers.csv", header=True, inferSchema=True)

print("Orders:", orders.count(), "rows | Suppliers:", suppliers.count(), "rows")
print(orders.columns)
orders.show(5)
suppliers.show()

Orders: 15 rows | Suppliers: 5 rows
['order_id', 'supplier_id', 'product_id', 'quantity', 'order_date', 'expected_date', 'delivery_date', 'status', 'delay_days', 'is_delayed', 'lead_time_days']
+--------+-----------+----------+--------+----------+-------------+-------------+---------+----------+----------+--------------+
|order_id|supplier_id|product_id|quantity|order_date|expected_date|delivery_date|   status|delay_days|is_delayed|lead_time_days|
+--------+-----------+----------+--------+----------+-------------+-------------+---------+----------+----------+--------------+
|       1|          1|         1|     200|2026-09-01|   2026-09-08|   2026-09-07|Delivered|         0|         0|           6.0|
|       2|          1|         2|     300|2026-09-02|   2026-09-09|   2026-09-12|Delivered|         3|         1|          10.0|
|       3|          2|         3|     150|2026-09-03|   2026-09-13|   2026-09-13|Delivered|         0|         0|          10.0|
|       4|          2|         4

**2.Filter delayed shipments.**

In [10]:
delayed_shipments = orders.filter(F.col("is_delayed") == 1)
delayed_shipments.show()
print("Delayed shipments:", delayed_shipments.count(), "rows")

+--------+-----------+----------+--------+----------+-------------+-------------+---------+----------+----------+--------------+
|order_id|supplier_id|product_id|quantity|order_date|expected_date|delivery_date|   status|delay_days|is_delayed|lead_time_days|
+--------+-----------+----------+--------+----------+-------------+-------------+---------+----------+----------+--------------+
|       2|          1|         2|     300|2026-09-02|   2026-09-09|   2026-09-12|Delivered|         3|         1|          10.0|
|       4|          2|         4|     100|2026-09-05|   2026-09-15|   2026-09-20|Delivered|         5|         1|          15.0|
|       6|          3|         6|     200|2026-09-08|   2026-09-13|   2026-09-16|Delivered|         3|         1|           8.0|
|       8|          4|        10|     250|2026-09-12|   2026-09-22|   2026-09-28|Delivered|         6|         1|          16.0|
|      10|          5|         9|     300|2026-09-18|   2026-09-28|   2026-10-02|Delivered|      

**3.Group by supplier and count delayed orders.**

In [9]:
result = delayed_shipments.groupBy("supplier_id").agg(F.count("*").alias("delayed_orders"))
result.show()

+-----------+--------------+
|supplier_id|delayed_orders|
+-----------+--------------+
|          1|             1|
|          3|             2|
|          5|             1|
|          4|             1|
|          2|             2|
+-----------+--------------+



**4.Save processed data to CSV or Parquet.**

In [11]:
import glob

result.coalesce(1).write.mode("overwrite").option("header", True).csv("output/delayed_orders_by_supplier")
result.write.mode("overwrite").parquet("output/delayed_orders_by_supplier_parquet")

csv_file = glob.glob("output/delayed_orders_by_supplier/part-*.csv")[0]
print(open(csv_file).read())

supplier_id,delayed_orders
1,1
3,2
5,1
4,1
2,2

